In [2]:
# ==============================================================================
# PART 1: ENVIRONMENT SETUP & INSTALLATIONS
# ==============================================================================
import os
import sys
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import clear_output

# 1.1 Check for Dataset
if not os.path.exists('data.csv'):
    print("Please upload your 'data.csv' file:")
    uploaded = files.upload()

print("Step 1: Installing dependencies (Java, Spark, Kafka)...")
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.0/kafka_2.13-3.6.0.tgz
!tar -xzf kafka_2.13-3.6.0.tgz

!pip install -q pyspark==3.5.0 kafka-python hmmlearn tensorflow scikit-learn

# Download Specific Kafka-Spark JARs
!wget -q https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar
!wget -q https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.5.0/kafka-clients-3.5.0.jar
!wget -q https://repo1.maven.org/maven2/org/apache/spark/spark-token-provider-kafka-0-10_2.12/3.5.0/spark-token-provider-kafka-0-10_2.12-3.5.0.jar
!wget -q https://repo1.maven.org/maven2/org/apache/commons/commons-pool2/2.11.1/commons-pool2-2.11.1.jar

# Set Environment Variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
sys.path.insert(0, f"{os.environ['SPARK_HOME']}/python")
sys.path.insert(0, f"{os.environ['SPARK_HOME']}/python/lib/py4j-0.10.9.7-src.zip")

# ==============================================================================
# PART 2: START KAFKA SERVICES
# ==============================================================================
print("Step 2: Starting Zookeeper and Kafka...")
!nohup kafka_2.13-3.6.0/bin/zookeeper-server-start.sh kafka_2.13-3.6.0/config/zookeeper.properties > zookeeper.log 2>&1 &
!nohup kafka_2.13-3.6.0/bin/kafka-server-start.sh kafka_2.13-3.6.0/config/server.properties > kafka.log 2>&1 &
time.sleep(15)

# Create Topics
!kafka_2.13-3.6.0/bin/kafka-topics.sh --create --bootstrap-server localhost:9092 --topic market_topic --partitions 1 --replication-factor 1 2>/dev/null
!kafka_2.13-3.6.0/bin/kafka-topics.sh --create --bootstrap-server localhost:9092 --topic sentiment_topic --partitions 1 --replication-factor 1 2>/dev/null
print("Kafka Services Online.")

# ==============================================================================
# PART 3: TRAIN ML MODELS (HMM & LSTM)
# ==============================================================================
from hmmlearn.hmm import GaussianHMM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

print("Step 3: Training Market Models...")
df = pd.read_csv('data.csv')

# 3.1 HMM Regime Detection
hmm_data = df[['d_return', 'volume_change']].fillna(0).values
hmm_model = GaussianHMM(n_components=3, n_iter=100).fit(hmm_data)

# 3.2 LSTM Price Prediction
features = ['open', 'high', 'low', 'close', 'volume']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features])

def create_sequences(data, seq_length=20):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 3]) # Target index 3 is 'close'
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(scaled_data)
lstm_model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(20, 5)),
    LSTM(50),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_train, y_train, epochs=2, batch_size=32, verbose=0)
print("Models Ready.")

# ==============================================================================
# PART 4: SPARK STREAMING & JOIN LOGIC
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, udf, expr, to_timestamp
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from kafka import KafkaProducer

jar_path = "/content/spark-sql-kafka-0-10_2.12-3.5.0.jar," \
           "/content/kafka-clients-3.5.0.jar," \
           "/content/spark-token-provider-kafka-0-10_2.12-3.5.0.jar," \
           "/content/commons-pool2-2.11.1.jar"

spark = SparkSession.builder \
    .appName("MarketSentimentPipeline") \
    .config("spark.jars", jar_path) \
    .getOrCreate()

# Define Schemas
m_schema = StructType([
    StructField("formatted_date", StringType()),
    StructField("open", DoubleType()), StructField("high", DoubleType()),
    StructField("low", DoubleType()), StructField("close", DoubleType()),
    StructField("volume", DoubleType())
])

s_schema = StructType([
    StructField("news_date", StringType()),
    StructField("sentiment_score", DoubleType()),
    StructField("headline", StringType())
])

# LSTM UDF
def predict_price(o, h, l, c, v):
    try:
        raw = np.array([[o, h, l, c, v]])
        s_raw = scaler.transform(raw)
        seq = np.repeat(s_raw[np.newaxis, :, :], 20, axis=1)
        p_scaled = lstm_model.predict(seq, verbose=0)
        dummy = np.zeros((1, 5))
        dummy[0, 3] = p_scaled[0, 0]
        return float(scaler.inverse_transform(dummy)[0, 3])
    except: return 0.0

predict_udf = udf(predict_price, DoubleType())

# Build Streams
m_stream = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "market_topic").load() \
    .selectExpr("CAST(value AS STRING)").select(from_json(col("value"), m_schema).alias("data")).select("data.*") \
    .withColumn("m_time", to_timestamp("formatted_date"))

s_stream = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "sentiment_topic").load() \
    .selectExpr("CAST(value AS STRING)").select(from_json(col("value"), s_schema).alias("data")).select("data.*") \
    .withColumn("s_time", to_timestamp("news_date"))

# Join and Predict
final_df = m_df.join(s_df, expr("m_time = s_time")) \
    .withColumn("lstm_prediction", predict_udf(col("open"), col("high"), col("low"), col("close"), col("volume")))

query = final_df.writeStream.queryName("live_analytics").outputMode("append").format("memory").start()

# ==============================================================================
# PART 5: LIVE DASHBOARD LOOP (FIXED)
# ==============================================================================
producer = KafkaProducer(bootstrap_servers=['localhost:9092'], value_serializer=lambda v: json.dumps(v).encode('utf-8'))
headlines = ["Fed Rate Hike", "Tech Earnings Beat", "Global Trade War", "Inflation Eases", "AI Breakthrough"]

print("Starting live dashboard...")
try:
    for i in range(40):
        # 1. Send Market and Sentiment Data
        row = df.iloc[i + 200]
        dt = str(row['formatted_date'])

        # Correctly structured market dictionary
        market_msg = {
            "formatted_date": dt,
            "open": float(row['open']), "high": float(row['high']),
            "low": float(row['low']), "close": float(row['close']),
            "volume": float(row['volume'])
        }
        producer.send('market_topic', market_msg)

        # Random sentiment for that day
        sentiment_msg = {
            "news_date": dt,
            "sentiment_score": float(np.random.uniform(-1, 1)),
            "headline": np.random.choice(headlines)
        }
        producer.send('sentiment_topic', sentiment_msg)

        producer.flush()
        time.sleep(2.5) # Allow Spark to process

        # 2. Update Visualization
        try:
            res_df = spark.sql("""
                SELECT formatted_date, close, lstm_prediction, sentiment_score,
                AVG(close) OVER (ORDER BY formatted_date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) as sma_5
                FROM live_analytics ORDER BY formatted_date ASC
            """).toPandas()

            if not res_df.empty:
                clear_output(wait=True)
                plt.figure(figsize=(12, 6))
                plt.plot(res_df['formatted_date'], res_df['close'], label='Market Close', marker='o', color='blue')
                plt.plot(res_df['formatted_date'], res_df['lstm_prediction'], label='LSTM Forecast', linestyle='--', color='orange')
                plt.plot(res_df['formatted_date'], res_df['sma_5'], label='SMA-5', color='green', alpha=0.6)
                plt.title(f"Live Pipeline (Iteration {i+1}) | Sentiment Score: {res_df['sentiment_score'].iloc[-1]:.2f}")
                plt.xticks(rotation=45)
                plt.legend()
                plt.grid(True, alpha=0.2)
                plt.show()
        except:
            pass # Wait for first record to hit memory

except KeyboardInterrupt:
    print("Stopped.")
finally:
    query.stop()
    print("Pipeline Shutdown.")

Step 1: Installing dependencies (Java, Spark, Kafka)...
Step 2: Starting Zookeeper and Kafka...
Error while executing topic command : Topic 'market_topic' already exists.
Error while executing topic command : Topic 'sentiment_topic' already exists.
Kafka Services Online.
Step 3: Training Market Models...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Models Ready.
Starting live dashboard...
Pipeline Shutdown.
